# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Diagnóstico intermedio de clusters (control de calidad post-clustering)

**Flujo de trabajo:**
1. Cargar objeto clusterizado
2. Cuantificar dominancia de muestra por cluster
3. Evaluar la señal QC (mitocondria / genes detectados) de forma relativa y conjunta
4. Comprobar coherencia biológica con marcadores canónicos, independiente del archivo de anotación oficial
5. Tabla de decisión por cluster
6. Guardar objeto con el diagnóstico


# · Importaciones y configuración

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import yaml

sc.settings.verbosity = 2

# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

INPUT_PATH  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['05_clustered']}/{PARAMS['outputs']['clustered_h5ad']}"
OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['06_cluster_diagnostics']}"
OUTPUT_PATH = f"{OUTPUT_DIR}/{PARAMS['outputs']['clustered_diagnostics_h5ad']}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables']['06_cluster_diagnostics']}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures']['06_cluster_diagnostics']}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR

BATCH_KEY = PARAMS["pipeline"]["batch_key"]
CONDITION_KEY = "condition"
MAIN_KEY      = "leiden_r0.8"
N_PCS         = 20

# Umbrales del diagnóstico
SAMPLE_FOLD_ENRICHMENT = 3.0   # una muestra con >3x su proporción global en el cluster
SAMPLE_MAX_PROP        = 0.30  # una muestra que supere el 30% del cluster
MIN_SAMPLES_IN_CLUSTER = 3     # clusters presentes en <3 muestras se tratan como sospechosos por defecto
ZSCORE_QC_THRESHOLD    = 1.5   # umbral robusto (MAD) para mito alta / genes bajos conjuntos
MARKER_SCORE_MIN       = 0.10  # score mínimo de sc.tl.score_genes para considerar una identidad clara

print("Configuración cargada")

Configuración cargada


In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
print(f"   {adata.n_obs:,} células × {adata.n_vars:,} genes")

assert MAIN_KEY in adata.obs.columns, f"ERROR: '{MAIN_KEY}' no encontrado."
n_clusters = adata.obs[MAIN_KEY].nunique()
cluster_ids = sorted(adata.obs[MAIN_KEY].unique(), key=int)
print(f"   {n_clusters} clusters detectados ({MAIN_KEY})")

   43,388 células × 33,538 genes
   21 clusters detectados (leiden_r0.8)


# · 1 — Dominancia de muestra por cluster

In [ ]:
global_sample_prop = adata.obs[BATCH_KEY].value_counts(normalize=True)

dominance_rows = []
for cl in cluster_ids:
    sub = adata.obs[adata.obs[MAIN_KEY] == cl]
    n_cells = len(sub)
    cluster_prop = sub[BATCH_KEY].value_counts(normalize=True)
    n_samples_present = (sub[BATCH_KEY].value_counts() > 0).sum()

    top_sample   = cluster_prop.index[0]
    top_prop     = cluster_prop.iloc[0]
    fold_enrich  = top_prop / global_sample_prop[top_sample]

    flag = (fold_enrich > SAMPLE_FOLD_ENRICHMENT and top_prop > SAMPLE_MAX_PROP) \
           or (n_samples_present < MIN_SAMPLES_IN_CLUSTER)

    dominance_rows.append({
        'cluster': cl, 'n_cells': n_cells,
        'top_sample': top_sample, 'top_sample_prop': round(top_prop, 3),
        'fold_enrichment': round(fold_enrich, 2),
        'n_samples_present': n_samples_present,
        'flag_sample_dominance': flag
    })

df_dominance = pd.DataFrame(dominance_rows)
print(df_dominance.to_string(index=False))

flagged = df_dominance[df_dominance['flag_sample_dominance']]['cluster'].tolist()
if flagged:
    print(f"\n     Clusters con posible dominancia de muestra: {flagged}")
else:
    print("\n    Ningún cluster muestra un enriquecimiento de muestra relevante.")

df_dominance.to_csv(f"{TABLES_DIR}/diag_sample_dominance.csv", index=False)


cluster  n_cells      top_sample  top_sample_prop  fold_enrichment  n_samples_present  flag_sample_dominance
      0     4865 GSM6614350_HC-3            0.123             1.74                 18                  False
      1     4673 GSM6614365_CD-6            0.211             2.07                 18                  False
      2     4436 GSM6614357_UC-4            0.111             1.65                 18                  False
      3     4162 GSM6614355_UC-2            0.142             2.51                 18                  False
      4     3675 GSM6614355_UC-2            0.138             2.44                 18                  False
      5     3451 GSM6614357_UC-4            0.131             1.94                 18                  False
      6     2939 GSM6614358_UC-5            0.217             3.64                 18                  False
      7     2478 GSM6614352_HC-5            0.174             2.45                 18                  False
      8     2421 GS

# · 2 — Señal QC conjunta (mitocondria × genes)


In [ ]:
if 'n_genes_by_counts' not in adata.obs.columns:
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

stats_rows = []
for cl in cluster_ids:
    sub = adata.obs[adata.obs[MAIN_KEY] == cl]
    stats_rows.append({
        'cluster': cl,
        'med_genes': sub['n_genes_by_counts'].median(),
        'med_mito_pct': sub['pct_counts_mt'].median(),
    })
qc_df = pd.DataFrame(stats_rows)

def robust_z(x):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) * 1.4826  # escalado a equivalente de SD bajo normalidad
    return (x - med) / mad if mad > 0 else pd.Series(np.zeros(len(x)), index=x.index)

qc_df['z_mito']  = robust_z(qc_df['med_mito_pct'])
qc_df['z_genes'] = robust_z(qc_df['med_genes'])
qc_df['flag_qc_joint_outlier'] = (qc_df['z_mito'] > ZSCORE_QC_THRESHOLD) & \
                                  (qc_df['z_genes'] < -ZSCORE_QC_THRESHOLD)

print(qc_df.to_string(index=False, float_format='%.2f'))

flagged_qc = qc_df[qc_df['flag_qc_joint_outlier']]['cluster'].tolist()
if flagged_qc:
    print(f"\n     Clusters con firma de baja calidad (mito alta + genes bajos, "
          f"ambos > {ZSCORE_QC_THRESHOLD} MAD respecto al resto de clusters): {flagged_qc}")
else:
    print("\n    Ningún cluster muestra la firma conjunta de baja calidad. ")

qc_df.to_csv(f"{TABLES_DIR}/diag_qc_joint.csv", index=False)


cluster  med_genes  med_mito_pct  z_mito  z_genes  flag_qc_joint_outlier
      0    1207.00         10.29   -0.14     0.00                  False
      1     774.00         17.99    0.88    -0.99                  False
      2    1248.50          8.34   -0.40     0.10                  False
      3    1513.00          3.39   -1.06     0.70                  False
      4    1284.00          2.49   -1.18     0.18                  False
      5     920.00          9.46   -0.25    -0.66                  False
      6    1394.00          2.34   -1.20     0.43                  False
      7    1501.00         11.70    0.04     0.67                  False
      8     963.00         29.71    2.43    -0.56                  False
      9    1240.50         10.62   -0.10     0.08                  False
     10     864.00         24.17    1.70    -0.79                  False
     11    1279.50          8.66   -0.36     0.17                  False
     12     762.00         11.37    0.00    -1.02  

# · 3 — Coherencia con marcadores canónicos


In [ ]:
markers_canonical = {
    "Epitelial":   ["EPCAM", "KRT20", "MUC2"],
    "Goblet":      ["MUC2", "TFF3"],
    "T cell":      ["CD3D", "CD3E", "CD8A", "CD4"],
    "B cell":      ["CD79A", "MS4A1"],
    "Plasma":      ["IGHA1", "MZB1"],
    "Macrophage":  ["C1QA", "CD68"],
    "Monocyte":    ["LYZ", "S100A8"],
    "Fibroblast":  ["COL1A1", "DCN"],
    "Endothelial": ["VWF", "PECAM1"],
    "Mast":        ["TPSAB1", "CPA3"],
    "Tuft":        ["POU2F3", "TRPM5"]
}

for lineage, genes in markers_canonical.items():
    genes_present = [g for g in genes if g in adata.var_names]
    if genes_present:
        sc.tl.score_genes(adata, gene_list=genes_present, score_name=f"score_{lineage}")

score_cols = [f"score_{l}" for l in markers_canonical if f"score_{l}" in adata.obs.columns]

marker_rows = []
for cl in cluster_ids:
    sub = adata.obs[adata.obs[MAIN_KEY] == cl]
    mean_scores = sub[score_cols].mean()
    best_lineage = mean_scores.idxmax().replace('score_', '')
    best_score   = mean_scores.max()
    marker_rows.append({
        'cluster': cl,
        'best_lineage_match': best_lineage,
        'best_score': round(best_score, 3),
        'flag_weak_identity': best_score < MARKER_SCORE_MIN
    })

df_markers = pd.DataFrame(marker_rows)
print(df_markers.to_string(index=False))

weak_id = df_markers[df_markers['flag_weak_identity']]['cluster'].tolist()
if weak_id:
    print(f"\n     Clusters sin identidad de linaje clara (score < {MARKER_SCORE_MIN}): {weak_id}")
else:
    print("\n    Todos los clusters muestran al menos un linaje con score positivo.")

df_markers.to_csv(f"{TABLES_DIR}/diag_marker_identity.csv", index=False)


computing score 'score_Epitelial'
    finished (0:00:00)
computing score 'score_Goblet'
    finished (0:00:00)
computing score 'score_T cell'
    finished (0:00:00)
computing score 'score_B cell'
    finished (0:00:00)
computing score 'score_Plasma'
    finished (0:00:00)
computing score 'score_Macrophage'
    finished (0:00:00)
computing score 'score_Monocyte'
    finished (0:00:00)
computing score 'score_Fibroblast'
    finished (0:00:00)
computing score 'score_Endothelial'
    finished (0:00:00)
computing score 'score_Mast'
    finished (0:00:00)
computing score 'score_Tuft'
    finished (0:00:00)
cluster best_lineage_match  best_score  flag_weak_identity
      0             T cell       1.341               False
      1               Tuft      -0.013                True
      2             T cell       1.461               False
      3             Plasma     841.936               False
      4             Plasma     691.913               False
      5             T cell       0.636

# · Tabla de decisión por cluster

Se combinan las tres comprobaciones en una única tabla con el razonamiento aplicado.


In [ ]:
decision_df = (df_dominance[['cluster', 'n_cells', 'flag_sample_dominance']]
               .merge(qc_df[['cluster', 'flag_qc_joint_outlier']], on='cluster')
               .merge(df_markers[['cluster', 'best_lineage_match', 'flag_weak_identity']], on='cluster'))

def build_reason(row):
    reasons = []
    if row['flag_sample_dominance']:
        reasons.append("dominancia de muestra (posible batch residual)")
    if row['flag_qc_joint_outlier']:
        reasons.append("firma conjunta (posible baja calidad)")
    if row['flag_weak_identity']:
        reasons.append("sin identidad de linaje clara (score de marcadores bajo)")
    return "; ".join(reasons) if reasons else "sin hallazgos"

decision_df['n_flags'] = decision_df[['flag_sample_dominance', 'flag_qc_joint_outlier',
                                       'flag_weak_identity']].sum(axis=1)
decision_df['reason']  = decision_df.apply(build_reason, axis=1)

def decide(row):
    if row['n_flags'] == 0:
        return 'OK'
    if row['n_flags'] == 1 and row['flag_sample_dominance'] and not row['flag_weak_identity']:
        # Dominancia de muestra pero con identidad biológica reconocible
        return 'REVISAR_COMPOSICION'
    if row['n_flags'] >= 2:
        return 'SOSPECHOSO_BAJA_CALIDAD'
    return 'REVISAR'

decision_df['decision'] = decision_df.apply(decide, axis=1)

cols = ['cluster', 'n_cells', 'best_lineage_match', 'n_flags', 'decision', 'reason']
print(decision_df[cols].to_string(index=False))

print(f"\n   Resumen: {(decision_df['decision']=='OK').sum()} OK | "
      f"{(decision_df['decision']=='REVISAR_COMPOSICION').sum()} a revisar (composición) | "
      f"{(decision_df['decision']=='REVISAR').sum()} a revisar | "
      f"{(decision_df['decision']=='SOSPECHOSO_BAJA_CALIDAD').sum()} sospechosos de baja calidad")

decision_df.to_csv(f"{TABLES_DIR}/cluster_decision_table.csv", index=False)

# Propagar la decisión a nivel de célula para que 06 la use al construir el consenso
cluster_to_decision = dict(zip(decision_df['cluster'], decision_df['decision']))
cluster_to_reason   = dict(zip(decision_df['cluster'], decision_df['reason']))
adata.obs['cluster_qc_status'] = adata.obs[MAIN_KEY].map(cluster_to_decision)
adata.obs['cluster_qc_reason'] = adata.obs[MAIN_KEY].map(cluster_to_reason)


cluster  n_cells best_lineage_match  n_flags            decision                                                   reason
      0     4865             T cell        0                  OK                                            sin hallazgos
      1     4673               Tuft        1             REVISAR sin identidad de linaje clara (score de marcadores bajo)
      2     4436             T cell        0                  OK                                            sin hallazgos
      3     4162             Plasma        0                  OK                                            sin hallazgos
      4     3675             Plasma        0                  OK                                            sin hallazgos
      5     3451             T cell        0                  OK                                            sin hallazgos
      6     2939             B cell        0                  OK                                            sin hallazgos
      7     2478        

# · Guardar objeto con diagnóstico

Este objeto es la entrada de 06. No se elimina ninguna célula ni cluster en
este paso, solo se documenta.

In [ ]:
print(f"\n→ Guardando en {OUTPUT_PATH} ...")
adata.write_h5ad(OUTPUT_PATH, compression='gzip')


→ Guardando en /content/drive/MyDrive/IBD_TFM/data/interim/06_cluster_diagnostics/IBD_clustered_diagnostics.h5ad ...
